# 02 - Exploratory Data Analysis & Visualizations

This notebook generates all visualizations for the hackathon report.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_palette('husl')

# Paths
BASE_DIR = Path('..')
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
FIGURES_DIR = BASE_DIR / 'outputs' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Configuration complete')

In [ ]:
# Load processed data
df_enrol = pd.read_parquet(PROCESSED_DIR / 'enrolment_cleaned.parquet')
df_bio = pd.read_parquet(PROCESSED_DIR / 'biometric_cleaned.parquet')
df_demo = pd.read_parquet(PROCESSED_DIR / 'demographic_cleaned.parquet')

print(f"Enrolment: {len(df_enrol):,} rows")
print(f"Biometric: {len(df_bio):,} rows")
print(f"Demographic: {len(df_demo):,} rows")

## 2.1 Univariate Analysis

In [ ]:
# Distribution of daily enrolments per pincode
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Enrolment distribution
axes[0].hist(df_enrol['total_enrol'], bins=50, edgecolor='white', alpha=0.7)
axes[0].set_xlabel('Total Enrolments per Pincode-Day')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Daily Enrolments')
axes[0].axvline(df_enrol['total_enrol'].median(), color='red', linestyle='--', label=f"Median: {df_enrol['total_enrol'].median():.0f}")
axes[0].legend()

# Biometric distribution
axes[1].hist(df_bio['total_bio'], bins=50, edgecolor='white', alpha=0.7, color='orange')
axes[1].set_xlabel('Total Biometric Updates per Pincode-Day')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Daily Biometric Updates')
axes[1].axvline(df_bio['total_bio'].median(), color='red', linestyle='--', label=f"Median: {df_bio['total_bio'].median():.0f}")
axes[1].legend()

# Demographic distribution
axes[2].hist(df_demo['total_demo'], bins=50, edgecolor='white', alpha=0.7, color='green')
axes[2].set_xlabel('Total Demographic Updates per Pincode-Day')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Distribution of Daily Demographic Updates')
axes[2].axvline(df_demo['total_demo'].median(), color='red', linestyle='--', label=f"Median: {df_demo['total_demo'].median():.0f}")
axes[2].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Age group composition - Enrolment
age_totals = {
    'Age 0-5': df_enrol['age_0_5'].sum(),
    'Age 5-17': df_enrol['age_5_17'].sum(),
    'Age 18+': df_enrol['age_18_greater'].sum()
}

fig, ax = plt.subplots(figsize=(8, 8))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
wedges, texts, autotexts = ax.pie(
    age_totals.values(), 
    labels=age_totals.keys(),
    autopct='%1.1f%%',
    colors=colors,
    explode=(0.02, 0.02, 0.02),
    shadow=True
)
ax.set_title('Enrolment Distribution by Age Group', fontsize=14, fontweight='bold')
plt.savefig(FIGURES_DIR / '02_age_distribution_pie.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.2 State-wise Analysis

In [ ]:
# Top 15 states by enrolment
state_enrol = df_enrol.groupby('state')['total_enrol'].sum().sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(state_enrol.index, state_enrol.values, color=sns.color_palette('viridis', len(state_enrol)))
ax.set_xlabel('Total Enrolments')
ax.set_title('Top 15 States by Aadhaar Enrolment Volume', fontsize=14, fontweight='bold')

# Add value labels
for bar, val in zip(bars, state_enrol.values):
    ax.text(val + max(state_enrol.values)*0.01, bar.get_y() + bar.get_height()/2, 
            f'{val:,.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_top_states_enrolment.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# State-wise comparison: Enrolment vs Updates
state_enrol_total = df_enrol.groupby('state')['total_enrol'].sum()
state_bio_total = df_bio.groupby('state')['total_bio'].sum()
state_demo_total = df_demo.groupby('state')['total_demo'].sum()

# Combine and get top 10
state_comparison = pd.DataFrame({
    'Enrolment': state_enrol_total,
    'Biometric': state_bio_total,
    'Demographic': state_demo_total
}).fillna(0)

top_states = state_comparison.sum(axis=1).sort_values(ascending=False).head(10).index
state_comparison_top = state_comparison.loc[top_states]

state_comparison_top.plot(kind='bar', figsize=(12, 6), width=0.8)
plt.title('Top 10 States: Enrolment vs Updates Comparison', fontsize=14, fontweight='bold')
plt.xlabel('State')
plt.ylabel('Total Count')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Activity Type')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_state_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.3 Time Series Analysis

In [ ]:
# Daily trends
daily_enrol = df_enrol.groupby('date')['total_enrol'].sum()
daily_bio = df_bio.groupby('date')['total_bio'].sum()
daily_demo = df_demo.groupby('date')['total_demo'].sum()

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(daily_enrol.index, daily_enrol.values, color='#3498db', linewidth=1.5)
axes[0].fill_between(daily_enrol.index, daily_enrol.values, alpha=0.3, color='#3498db')
axes[0].set_ylabel('Enrolments')
axes[0].set_title('Daily Aadhaar Activity Trends', fontsize=14, fontweight='bold')
axes[0].legend(['Enrolments'])

axes[1].plot(daily_bio.index, daily_bio.values, color='#e74c3c', linewidth=1.5)
axes[1].fill_between(daily_bio.index, daily_bio.values, alpha=0.3, color='#e74c3c')
axes[1].set_ylabel('Biometric Updates')
axes[1].legend(['Biometric Updates'])

axes[2].plot(daily_demo.index, daily_demo.values, color='#2ecc71', linewidth=1.5)
axes[2].fill_between(daily_demo.index, daily_demo.values, alpha=0.3, color='#2ecc71')
axes[2].set_ylabel('Demographic Updates')
axes[2].set_xlabel('Date')
axes[2].legend(['Demographic Updates'])

plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_daily_trends.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Day of week patterns
dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow_enrol = df_enrol.groupby('day_of_week')['total_enrol'].mean()

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(dow_names, dow_enrol.values, color=sns.color_palette('coolwarm', 7))
ax.set_xlabel('Day of Week')
ax.set_ylabel('Average Enrolments')
ax.set_title('Average Daily Enrolments by Day of Week', fontsize=14, fontweight='bold')

# Highlight weekend
bars[5].set_color('#ff6b6b')
bars[6].set_color('#ff6b6b')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '06_day_of_week.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.4 Heatmaps

In [ ]:
# State x Month heatmap for enrolment
state_month = df_enrol.pivot_table(
    values='total_enrol', 
    index='state', 
    columns='month', 
    aggfunc='sum'
).fillna(0)

# Get top 20 states
top_states = state_month.sum(axis=1).sort_values(ascending=False).head(20).index
state_month_top = state_month.loc[top_states]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(state_month_top, cmap='YlOrRd', annot=True, fmt='.0f', 
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Total Enrolments'})
ax.set_title('State x Month Enrolment Heatmap (Top 20 States)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('State')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '07_state_month_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Age group trends over time
age_time = df_enrol.groupby('date')[['age_0_5', 'age_5_17', 'age_18_greater']].sum()

fig, ax = plt.subplots(figsize=(14, 6))
age_time.plot(ax=ax, linewidth=2)
ax.set_xlabel('Date')
ax.set_ylabel('Enrolments')
ax.set_title('Enrolment Trends by Age Group', fontsize=14, fontweight='bold')
ax.legend(['Age 0-5', 'Age 5-17', 'Age 18+'], title='Age Group')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '08_age_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.5 Correlation Analysis

In [ ]:
# Correlation between age groups in enrolment
corr_cols = ['age_0_5', 'age_5_17', 'age_18_greater', 'total_enrol']
corr_matrix = df_enrol[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, ax=ax, fmt='.2f')
ax.set_title('Correlation Matrix: Enrolment Age Groups', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '09_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.6 District-level Analysis

In [ ]:
# Top 20 districts by enrolment
district_enrol = df_enrol.groupby(['state', 'district'])['total_enrol'].sum().sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(12, 8))
district_labels = [f"{d[1]} ({d[0][:3]})" for d in district_enrol.index]
bars = ax.barh(district_labels[::-1], district_enrol.values[::-1], 
               color=sns.color_palette('plasma', 20)[::-1])
ax.set_xlabel('Total Enrolments')
ax.set_title('Top 20 Districts by Enrolment Volume', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '10_top_districts.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.7 Summary Statistics

In [ ]:
# Generate summary table
summary = pd.DataFrame({
    'Metric': ['Total Records', 'Date Range', 'States Covered', 'Districts Covered', 
               'Total Enrolments', 'Total Bio Updates', 'Total Demo Updates'],
    'Value': [
        f"{len(df_enrol) + len(df_bio) + len(df_demo):,}",
        f"{df_enrol['date'].min().strftime('%Y-%m-%d')} to {df_enrol['date'].max().strftime('%Y-%m-%d')}",
        df_enrol['state'].nunique(),
        df_enrol['district'].nunique(),
        f"{df_enrol['total_enrol'].sum():,.0f}",
        f"{df_bio['total_bio'].sum():,.0f}",
        f"{df_demo['total_demo'].sum():,.0f}"
    ]
})

print("\n=== Data Summary ===")
display(summary)

In [ ]:
print(f"\n✅ Visualization generation complete!")
print(f"Figures saved to: {FIGURES_DIR}")
print(f"Total figures: {len(list(FIGURES_DIR.glob('*.png')))}")